# 🔍 Scalable PDF Ingestion — Qdrant Cloud
**Branch: `scalable-ingestion`**

This notebook indexes PDFs stored in your Google Drive into a **Qdrant Cloud** persistent vector store. Run it once (or resume it) to index all your PDFs. The Streamlit query app (`app_cloud.py`) then queries Qdrant Cloud directly — no local vector DB needed.

**Capacity tested:** 1,635 PDFs / 3.44 GB  
**Resumable:** Yes — a checkpoint JSON is saved to Drive after each batch  
**Duplicate-safe:** Yes — SHA-256 fingerprinting skips already-indexed files

## 📋 Prerequisites

### 1. Qdrant Cloud Account
1. Go to [https://cloud.qdrant.io](https://cloud.qdrant.io) and create a free account
2. Create a new **Cluster** (free 1 GB tier is available)
3. From your cluster dashboard, copy:
   - **Cluster URL** (e.g. `https://xxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx.us-east4-0.gcp.cloud.qdrant.io:6333`)
   - **API Key** (under Data Access Control / API Keys)

### 2. Google Drive — Upload your PDFs
If your PDFs are not yet in Drive:
1. Open [drive.google.com](https://drive.google.com)
2. Create a new folder (e.g. `pdfs`) inside `My Drive`
3. Upload your PDF files into that folder
4. The full path will be `/content/drive/MyDrive/pdfs` after mounting

### 3. Set credentials in Cell 3 below
Fill in your Qdrant Cloud credentials and the Drive folder path before running.

In [ ]:
# ============================================================
# STEP 1 — Fill in your credentials before running this cell
# ============================================================

import os

# --- Qdrant Cloud credentials ---
# Get these from https://cloud.qdrant.io → Clusters
os.environ["QDRANT_URL"]             = "https://your-cluster-url.cloud.qdrant.io:6333"  # <-- replace
os.environ["QDRANT_API_KEY"]         = "your-qdrant-api-key-here"                      # <-- replace
os.environ["QDRANT_COLLECTION_NAME"] = "app_collection"                                # <-- replace if different

# --- Google Drive folder containing your PDFs ---
# After mounting Drive (Cell 5), your files will be at:
#   /content/drive/MyDrive/<folder-name>/
DRIVE_PDF_FOLDER = "/content/drive/MyDrive/pdfs"               # <-- replace if different

# --- Checkpoint file location (saved to Drive so it survives Colab restarts) ---
CHECKPOINT_PATH = "/content/drive/MyDrive/rag_checkpoint/ingestion_checkpoint.json"

# --- Collection settings (must match app_cloud.py) ---
COLLECTION_NAME        = os.environ["QDRANT_COLLECTION_NAME"]
CHUNK_SIZE             = 1000
CHUNK_OVERLAP          = 200
CHUNK_BATCH_SIZE       = 100
EMBEDDING_MODEL        = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_NAMESPACE    = "all-MiniLM-L6-v2"
EMBED_CACHE_DIR        = "/content/embedding_cache"
PROGRESS_REPORT_INTERVAL = 50

print("✅ Configuration set")
print(f"   PDF folder  : {DRIVE_PDF_FOLDER}")
print(f"   Checkpoint  : {CHECKPOINT_PATH}")
print(f"   Collection  : {COLLECTION_NAME}")


In [ ]:
# Install dependencies for Qdrant Cloud ingestion
!pip install -q \
    qdrant-client \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-core \
    langchain-classic \
    langchain-qdrant \
    langchain-huggingface \
    sentence-transformers \
    pypdf

print("✅ Dependencies installed")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

pdf_folder = Path(DRIVE_PDF_FOLDER)
if not pdf_folder.exists():
    print(f"❌ Folder not found: {pdf_folder}")
    print("   Create the folder in Google Drive and upload your PDFs, then re-run this cell.")
else:
    pdf_files = sorted(pdf_folder.glob("*.pdf"))
    total_size_gb = sum(f.stat().st_size for f in pdf_files) / (1024**3)
    print(f"✅ Drive mounted")
    print(f"   PDFs found : {len(pdf_files)}")
    print(f"   Total size : {total_size_gb:.2f} GB")

In [ ]:
# =============================================================================
# INLINE MODULES — All logic is self-contained here; no local .py imports
# =============================================================================

import hashlib
import json
import os
import tempfile
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, List, Optional

import pypdf
import torch
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.http import models as rest


# =============================================================================
# === MODULE: Checkpoint ===
# =============================================================================

def load_checkpoint(path: str) -> Dict[str, Any]:
    checkpoint_path = Path(path)
    if not checkpoint_path.exists():
        return {"indexed_files": {}, "runs": []}
    try:
        with checkpoint_path.open("r", encoding="utf-8") as fh:
            data = json.load(fh)
        if "indexed_files" not in data:
            data["indexed_files"] = {}
        if "runs" not in data:
            data["runs"] = []
        return data
    except (json.JSONDecodeError, OSError):
        print(f"⚠️  Checkpoint at {path} is corrupt or unreadable — starting fresh.")
        return {"indexed_files": {}, "runs": []}


def save_checkpoint(path: str, checkpoint: Dict[str, Any]) -> None:
    checkpoint_path = Path(path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_fd, tmp_name = tempfile.mkstemp(
        dir=str(checkpoint_path.parent), suffix=".tmp"
    )
    try:
        with os.fdopen(tmp_fd, "w", encoding="utf-8") as fh:
            json.dump(checkpoint, fh, indent=2, ensure_ascii=False)
        os.replace(tmp_name, str(checkpoint_path))
    except Exception:
        try:
            os.unlink(tmp_name)
        except OSError:
            pass
        raise


def is_already_indexed(checkpoint: Dict[str, Any], fingerprint: str) -> bool:
    return fingerprint in checkpoint.get("indexed_files", {})


def record_indexed_file(
    checkpoint: Dict[str, Any],
    fingerprint: str,
    file_name: str,
    size_bytes: int,
    chunk_count: int,
) -> None:
    checkpoint.setdefault("indexed_files", {})[fingerprint] = {
        "file_name": file_name,
        "size_bytes": size_bytes,
        "chunk_count": chunk_count,
        "indexed_at": datetime.now(timezone.utc).isoformat(),
    }


def append_run_summary(
    checkpoint: Dict[str, Any],
    files_indexed: int,
    files_skipped: int,
    files_failed: int,
    total_chunks: int,
) -> None:
    checkpoint.setdefault("runs", []).append({
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "files_indexed": files_indexed,
        "files_skipped": files_skipped,
        "files_failed": files_failed,
        "total_chunks": total_chunks,
    })


def checkpoint_summary(checkpoint: Dict[str, Any]) -> str:
    indexed = checkpoint.get("indexed_files", {})
    runs = checkpoint.get("runs", [])
    total_chunks = sum(
        entry.get("chunk_count", 0) for entry in indexed.values()
    )
    return (
        f"{len(indexed)} files indexed across {len(runs)} run(s); "
        f"{total_chunks:,} total chunks recorded"
    )


# =============================================================================
# === MODULE: Chunking ===
# =============================================================================

def build_text_splitter() -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " "],
        keep_separator=True,
    )


# =============================================================================
# === MODULE: Embeddings ===
# =============================================================================

@lru_cache(maxsize=1)
def load_embeddings() -> CacheBackedEmbeddings:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"   Using embedding device: {device}")
    base_embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True, "batch_size": 64},
    )
    store = LocalFileStore(EMBED_CACHE_DIR)
    cached_embedder = CacheBackedEmbeddings.from_bytes_store(
        base_embeddings,
        store,
        namespace=EMBEDDING_NAMESPACE,
        key_encoder="sha256",
    )
    return cached_embedder


# =============================================================================
# === MODULE: Fingerprinting ===
# =============================================================================

def fingerprint_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


# =============================================================================
# === MODULE: Ingestion helpers ===
# =============================================================================

def extract_pdf_documents(pdf_path: Path) -> list:
    """Fast in-memory PDF extraction with PyPDFLoader fallback."""
    try:
        reader = pypdf.PdfReader(str(pdf_path))
        docs = []
        for idx, page in enumerate(reader.pages):
            try:
                text = page.extract_text()
                if text and text.strip():
                    docs.append(Document(page_content=text, metadata={"page": idx + 1}))
            except Exception:
                pass
        if docs:
            return docs
    except Exception:
        pass
    try:
        return PyPDFLoader(str(pdf_path)).load()
    except Exception:
        return []


def _add_documents_in_batches(
    vector_db: QdrantVectorStore,
    chunks: list,
    batch_size: int,
) -> tuple:
    embedding_time = 0.0
    insertion_time = 0.0

    for start in range(0, len(chunks), batch_size):
        batch = chunks[start : start + batch_size]
        texts = [doc.page_content for doc in batch]
        point_ids = [
            str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.metadata["chunk_id"]))
            for doc in batch
        ]

        embed_start = time.time()
        embeddings_list = vector_db.embeddings.embed_documents(texts)
        embedding_time += time.time() - embed_start

        insert_start = time.time()
        points = [
            rest.PointStruct(
                id=pid,
                vector=vec,
                payload={
                    "page_content": doc.page_content,
                    "metadata": doc.metadata,
                },
            )
            for pid, vec, doc in zip(point_ids, embeddings_list, batch)
        ]
        vector_db.client.upsert(
            collection_name=vector_db.collection_name,
            points=points,
        )
        insertion_time += time.time() - insert_start

    return embedding_time, insertion_time


def _prepare_chunks(docs: list, fingerprint: str, file_name: str) -> list:
    splitter = build_text_splitter()
    chunks = splitter.split_documents(docs)
    total_chunks = len(chunks)

    for idx, chunk in enumerate(chunks):
        chunk_id = f"{fingerprint}_{idx}"
        page_number = chunk.metadata.get("page", 0)
        chunk.metadata.update({
            "filename": file_name,
            "source_pdf": file_name,
            "session_id": "colab_ingestion",
            "file_fingerprint": fingerprint,
            "chunk_id": chunk_id,
            "chunk_index": idx,
            "total_chunks": total_chunks,
            "page": page_number,
            "char_count": len(chunk.page_content),
        })

    return chunks


# =============================================================================
# === MODULE: Qdrant Cloud client ===
# =============================================================================

def get_cloud_store(embeddings: CacheBackedEmbeddings) -> QdrantVectorStore:
    client = QdrantClient(
        url=os.environ["QDRANT_URL"],
        api_key=os.environ.get("QDRANT_API_KEY") or None,
    )
    col_name = os.environ.get("QDRANT_COLLECTION_NAME", "app_collection")
    if not client.collection_exists(col_name):
        print(f"Creating Qdrant collection '{col_name}' (dim=384, distance=Cosine)...")
        client.create_collection(
            collection_name=col_name,
            vectors_config=rest.VectorParams(size=384, distance=rest.Distance.COSINE),
        )
    return QdrantVectorStore(
        client=client,
        collection_name=col_name,
        embedding=embeddings,
    )


# =============================================================================
# === MODULE: Metrics ===
# =============================================================================

@dataclass
class IndexingMetrics:
    started_at: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    pdf_count: int = 0
    dataset_size_mb: float = 0.0
    chunk_size: int = 0
    chunk_overlap: int = 0
    load_time: float = 0.0
    chunk_time: float = 0.0
    embedding_time: float = 0.0
    vector_insertion_time: float = 0.0
    total_chunks: int = 0
    processed_files: int = 0
    skipped_duplicates: int = 0
    empty_documents: int = 0
    failed_files: int = 0
    db_size_mb: float = 0.0
    errors: List[str] = field(default_factory=list)
    total_indexing_time: float = 0.0

    def as_dict(self) -> Dict[str, Any]:
        return {
            "started_at": self.started_at,
            "pdf_count": self.pdf_count,
            "dataset_size_mb": round(self.dataset_size_mb, 2),
            "chunk_size": self.chunk_size,
            "chunk_overlap": self.chunk_overlap,
            "load_time_s": round(self.load_time, 2),
            "chunk_time_s": round(self.chunk_time, 2),
            "embedding_time_s": round(self.embedding_time, 2),
            "vector_insertion_time_s": round(self.vector_insertion_time, 2),
            "total_indexing_time_s": round(self.total_indexing_time, 2),
            "total_chunks": self.total_chunks,
            "processed_files": self.processed_files,
            "skipped_duplicates": self.skipped_duplicates,
            "empty_documents": self.empty_documents,
            "failed_files": self.failed_files,
            "db_size_mb": round(self.db_size_mb, 2),
            "errors": self.errors,
        }

print("✅ All modules loaded")


In [ ]:
print("Connecting to Qdrant Cloud...")
try:
    store = get_cloud_store(embeddings)
    count = store.client.count(collection_name=store.collection_name).count
    print(f"✅ Qdrant Cloud connected — {count:,} vectors currently in collection '{store.collection_name}'")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Check your QDRANT_URL and QDRANT_API_KEY in Cell 3.")
    raise


In [ ]:
print("Loading ingestion checkpoint...")
checkpoint = load_checkpoint(CHECKPOINT_PATH)
print(f"   {checkpoint_summary(checkpoint)}")

already_indexed = len(checkpoint.get("indexed_files", {}))
remaining = len(pdf_files) - already_indexed
print(f"   PDFs in Drive  : {len(pdf_files)}")
print(f"   Already indexed: {already_indexed}")
print(f"   To process     : {remaining}")

In [ ]:
import time
import gc
import traceback

metrics = IndexingMetrics(
    pdf_count=len(pdf_files),
    dataset_size_mb=sum(f.stat().st_size for f in pdf_files) / (1024 * 1024),
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

files_indexed_this_run = 0
files_skipped_this_run = 0
files_failed_this_run = 0
total_chunks_this_run = 0

print(f"Starting ingestion of {len(pdf_files)} PDFs into Qdrant Cloud...")
print(f"({files_skipped_this_run + already_indexed} already indexed — will be skipped)")
print("-" * 60)

run_start = time.time()
store = get_cloud_store(embeddings)

CHECKPOINT_INTERVAL = 10  # Save checkpoint every 10 indexed files to minimize Drive write latency

for i, pdf_path in enumerate(pdf_files, 1):
    try:
        fingerprint = fingerprint_file(pdf_path)
    except Exception as exc:
        print(f"[{i}/{len(pdf_files)}] ❌ Cannot read {pdf_path.name}: {exc}")
        files_failed_this_run += 1
        metrics.failed_files += 1
        continue

    if is_already_indexed(checkpoint, fingerprint):
        files_skipped_this_run += 1
        metrics.skipped_duplicates += 1
        continue

    # --- Load PDF ---
    try:
        load_start = time.time()
        docs = extract_pdf_documents(pdf_path)
        metrics.load_time += time.time() - load_start

        if not docs:
            print(f"[{i}/{len(pdf_files)}] ⚠️  Empty: {pdf_path.name}")
            metrics.empty_documents += 1
            continue

        # --- Chunk ---
        chunk_start = time.time()
        chunks = _prepare_chunks(docs, fingerprint, pdf_path.name)
        metrics.chunk_time += time.time() - chunk_start

        if not chunks:
            print(f"[{i}/{len(pdf_files)}] ⚠️  No chunks: {pdf_path.name}")
            metrics.empty_documents += 1
            continue

        # --- Embed + insert into Qdrant Cloud ---
        try:
            embed_time, insert_time = _add_documents_in_batches(store, chunks, CHUNK_BATCH_SIZE)
        except Exception:
            # Re-establish connection on network drop
            store = get_cloud_store(embeddings)
            embed_time, insert_time = _add_documents_in_batches(store, chunks, CHUNK_BATCH_SIZE)

        metrics.embedding_time += embed_time
        metrics.vector_insertion_time += insert_time
        metrics.total_chunks += len(chunks)
        metrics.processed_files += 1
        total_chunks_this_run += len(chunks)
        files_indexed_this_run += 1

        # --- Update in-memory checkpoint ---
        record_indexed_file(checkpoint, fingerprint, pdf_path.name, pdf_path.stat().st_size, len(chunks))
        if files_indexed_this_run % CHECKPOINT_INTERVAL == 0:
            save_checkpoint(CHECKPOINT_PATH, checkpoint)

        print(f"[{i}/{len(pdf_files)}] ✅ {pdf_path.name} — {len(chunks)} chunks")

    except Exception as exc:
        files_failed_this_run += 1
        metrics.failed_files += 1
        error_msg = f"{pdf_path.name}: {exc}"
        metrics.errors.append(error_msg)
        print(f"[{i}/{len(pdf_files)}] ❌ FAILED: {pdf_path.name}")
        traceback.print_exc()

    finally:
        if i % 20 == 0:
            gc.collect()

    # --- Progress summary every N files ---
    if i % PROGRESS_REPORT_INTERVAL == 0:
        elapsed = time.time() - run_start
        rate = files_indexed_this_run / elapsed if elapsed > 0 else 0
        print(f"\n{'='*60}")
        print(f"Progress: {i}/{len(pdf_files)} files checked")
        print(f"  Indexed this run : {files_indexed_this_run}")
        print(f"  Skipped (dupes)  : {files_skipped_this_run}")
        print(f"  Failed           : {files_failed_this_run}")
        print(f"  Chunks so far    : {total_chunks_this_run:,}")
        print(f"  Rate             : {rate:.2f} files/s")
        print(f"  Elapsed          : {elapsed:.0f}s")
        print(f"{'='*60}\n")

# --- Final summary and save ---
metrics.total_indexing_time = time.time() - run_start
append_run_summary(checkpoint, files_indexed_this_run, files_skipped_this_run, files_failed_this_run, total_chunks_this_run)
save_checkpoint(CHECKPOINT_PATH, checkpoint)

print(f"\n{'='*60}")
print("INGESTION COMPLETE")
print(f"{'='*60}")
print(f"  Indexed this run : {files_indexed_this_run}")
print(f"  Skipped (dupes)  : {files_skipped_this_run}")
print(f"  Failed           : {files_failed_this_run}")
print(f"  Total chunks     : {total_chunks_this_run:,}")
print(f"  Total time       : {metrics.total_indexing_time:.1f}s")
print(f"  Embedding time   : {metrics.embedding_time:.1f}s")
print(f"  Insert time      : {metrics.vector_insertion_time:.1f}s")
if metrics.errors:
    print(f"\n  ERRORS ({len(metrics.errors)}):")
    for e in metrics.errors:
        print(f"    - {e}")
print(f"\n  Checkpoint saved to: {CHECKPOINT_PATH}")


In [ ]:
store = get_cloud_store(embeddings)
final_count = store.client.count(collection_name=store.collection_name).count
print(f"✅ Qdrant Cloud collection '{store.collection_name}' now contains {final_count:,} vectors")
print(f"   Checkpoint records {len(checkpoint['indexed_files'])} indexed files")


## ✅ Next Steps

Your PDFs are now indexed in Qdrant Cloud. To query them:

### Option A — Streamlit app (scalable-ingestion branch)
```bash
# From your local repo, on the scalable-ingestion branch:
streamlit run app_cloud.py
```

Set these in `.streamlit/secrets.toml`:
```toml
QDRANT_URL             = "https://your-cluster-url.cloud.qdrant.io:6333"
QDRANT_API_KEY         = "your-qdrant-api-key"
QDRANT_COLLECTION_NAME = "app_collection"
GROQ_API_KEY           = "your-groq-key"
```

### Option B — Streamlit Community Cloud
Deploy `app_cloud.py` from the `scalable-ingestion` branch and set the four secrets above in the Streamlit Cloud dashboard under App Settings → Secrets.

### Resume ingestion
If Colab disconnected mid-run, just re-run all cells from Cell 8 onward. The checkpoint will skip already-indexed files automatically.